# 02 · Mapping audit: every hand decision, both datasets

**What it does.** Lists, for ImageNet and LVIS together, every pooled class with its status and every anchor with its
origin, so curation is visible next to what WordNet alone gives.
- **Anchor origin:** `first_sense` (WordNet's first noun sense, no choice made), `other_sense` (another sense of the
  same word, chosen by hand), `added` (a different lemma, added by hand).
- **Class status:** `analyzed` (in the exclusive set the results use), `hyponym_only` (pooled but counted under a
  closer CDI word, or the word is outside valid129), `excluded` (removed by hand, reason given).

**Inputs:** `outputs/mapping/anchors_*.json` (notebook 01), the exclusive sets (notebook 05), LVIS annotations.
**Outputs:** `outputs/mapping/mapping_audit_{pairs,anchors,words}.csv` and the review page `mapping_audit.html`
(template: `notebooks/mapping_audit_template.html`).

In [1]:
# Standard setup shared by every notebook (each notebook is self-contained).
import json
import os
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist, squareform
from scipy.stats import false_discovery_control, pearsonr, rankdata, spearmanr, wilcoxon
from IPython.display import Image, Markdown, display

plt.rcParams["font.family"] = ["DejaVu Sans", "sans-serif"]
pd.set_option("display.max_colwidth", 120)


def show_images(paths, width=950):
    """Display saved figures (the files this notebook writes)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(ROOT)}`")); display(Image(filename=str(p), width=width))
        else:
            display(Markdown(f"*Missing figure:* `{p}`"))

In [2]:
# Paths. Run this notebook from analysis/comparison-datasets/notebooks/ (Jupyter's default working directory).
ROOT = Path.cwd().resolve().parent                     # analysis/comparison-datasets
assert (ROOT / "notebooks").is_dir(), "open this notebook from analysis/comparison-datasets/notebooks/"
REPO = ROOT.parents[1]
DATA_DIR = REPO / "data"
MANUSCRIPT_DIR = REPO / "analysis" / "manuscript-2026"
VEDB_DIR = REPO / "analysis" / "vedb-1612"
VOCAB_DIR = DATA_DIR / "vocabulary_lists"
VALID129_TXT = VOCAB_DIR / "included_categories_valid129.txt"

# Inputs written by other pipelines (read only).
EXEMPLAR_EMBED_ROOT = Path(os.environ.get("BV_EXEMPLAR_EMBED_ROOT",
                                          str(MANUSCRIPT_DIR / "exemplar_set_embeddings"))).expanduser()
BV_THINGS_VALID129 = EXEMPLAR_EMBED_ROOT / "valid129"       # manuscript Stage 0: BV + THINGS tables
VEDB_VALID129 = VEDB_DIR / "exemplar_set_embeddings" / "valid129"   # vedb-1612 Stage 0 (files use the "bv_" prefix)

# External data on ccn2.
IMAGENET_DIR = Path("/data2/j7yang/imagenet-1k/extracted")
IMAGENET_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
LVIS_ROOT = Path("/data2/j7yang/lvis")
LVIS_CROP_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")

# Outputs of this folder.
OUT = ROOT / "outputs"
MAPPING_OUT = OUT / "mapping"
MONTAGE_OUT = OUT / "montages"
EXEMPLAR_SETS = OUT / "exemplar_sets"
MODELS = ("clip", "dinov3")

## Helpers

In [3]:
from nltk.corpus import wordnet as wn

def hypernym_distances(s):
    """{ancestor synset: shortest hypernym-path distance}, including s itself at 0."""
    dist = {s: 0}
    frontier = [s]
    while frontier:
        nxt = []
        for x in frontier:
            for h in x.hypernyms() + x.instance_hypernyms():
                if h not in dist:
                    dist[h] = dist[x] + 1
                    nxt.append(h)
        frontier = nxt
    return dist


def anchor_origin(word: str, anchor: str) -> str:
    senses = [s.name() for s in wn.synsets(word, "n")]
    if senses and anchor == senses[0]:
        return "first_sense"
    if anchor in senses:
        return "other_sense"
    return "added"


def source_classes(dataset: str) -> pd.DataFrame:
    """id, name, synset, n_exemplars for every source class of the dataset."""
    if dataset == "imagenet":
        npz = __import__("numpy").load(MAPPING_OUT / "imagenet_class_embeddings_clip.npz")
        ids = [str(w) for w in npz["wnids"]]
        return pd.DataFrame({
            "source_id": ids,
            "source_synset": [wn.synset_from_pos_and_offset("n", int(w[1:])).name() for w in ids],
            "source_name": [wn.synset_from_pos_and_offset("n", int(w[1:])).lemma_names()[0] for w in ids],
            "n_exemplars": npz["counts"],
        })
    splits = [json.load(open(LVIS_ROOT / "annotations" / f"lvis_v1_{s}.json")) for s in ("train", "val")]
    cats = splits[1]["categories"]
    npz = __import__("numpy").load(MAPPING_OUT / "lvis_category_embeddings_clip.npz")
    n = dict(zip(npz["category_ids"].tolist(), npz["counts"].tolist()))  # embedded crops
    # Categories never embedded (e.g. excluded by hand): count their annotated boxes instead.
    boxes = pd.Series([a["category_id"] for d in splits for a in d["annotations"]]).value_counts().to_dict()
    n = {c["id"]: n.get(c["id"], boxes.get(c["id"], 0)) for c in cats}
    return pd.DataFrame({
        "source_id": [str(c["id"]) for c in cats],
        "source_synset": [c["synset"] for c in cats],
        "source_name": [c["name"] for c in cats],
        "n_exemplars": [n.get(c["id"], None) for c in cats],
    })


def audit(dataset: str, anchors: dict, exclude: dict, analyzed_csv: Path, id_col: str):
    src = source_classes(dataset)
    dists = {}
    for r in src.itertuples():
        try:
            dists[r.source_id] = hypernym_distances(wn.synset(r.source_synset))
        except Exception:  # e.g. LVIS stop_sign.n.01, not in NLTK WordNet
            continue
    analyzed = pd.read_csv(analyzed_csv)
    analyzed_pairs = set(zip(analyzed["cdi_category"], analyzed[id_col].astype(str)))
    analyzed_words = set(analyzed["cdi_category"])

    pair_rows, anchor_rows = [], []
    for word, anchor_names in anchors.items():
        for a in anchor_names:
            anchor_rows.append({"dataset": dataset, "cdi_category": word, "anchor_synset": a,
                                "anchor_origin": anchor_origin(word, a),
                                "anchor_definition": wn.synset(a).definition()})
        syns = [wn.synset(a) for a in anchor_names]
        excl = exclude.get(word, {})
        for r in src.itertuples():
            d = dists.get(r.source_id)
            if d is None:
                continue
            hits = [(d[s], s) for s in syns if s in d]
            if not hits:
                continue
            dist, a = min(hits, key=lambda t: t[0])
            if r.source_synset in excl:
                status, note = "excluded", excl[r.source_synset]
            elif (word, r.source_id) in analyzed_pairs:
                status, note = "analyzed", ""
            elif word in analyzed_words:
                closer = analyzed.loc[analyzed[id_col].astype(str) == r.source_id, "cdi_category"].tolist()
                status, note = "hyponym_only", f"assigned to closer CDI word: {', '.join(closer)}" if closer else ""
            else:
                status, note = "hyponym_only", "CDI word not in analyzed set"
            pair_rows.append({
                "dataset": dataset, "cdi_category": word, "source_id": r.source_id,
                "source_name": r.source_name, "source_synset": r.source_synset,
                "source_definition": wn.synset(r.source_synset).definition(),
                "n_exemplars": r.n_exemplars, "anchor_synset": a.name(),
                "anchor_origin": anchor_origin(word, a.name()), "hypernym_distance": dist,
                "status": status, "note": note,
            })
    return pd.DataFrame(pair_rows), pd.DataFrame(anchor_rows)

## Audit tables and review page

In [4]:
def build_audit_tables():
    A = {ds: json.loads((MAPPING_OUT / f"anchors_{ds}.json").read_text()) for ds in ("imagenet", "lvis")}  # written by notebook 01
    cdi = pd.read_csv(VOCAB_DIR / "cdi_concrete_nouns_295_with_semantic.csv")
    valid129 = set((VOCAB_DIR / "included_categories_valid129.txt").read_text().split())
    valid85 = set((VOCAB_DIR / "included_categories_valid85.txt").read_text().split())

    p_in, a_in = audit("imagenet", A["imagenet"]["anchors"], A["imagenet"]["exclude"],
                       EXEMPLAR_SETS / "imagenet_exclusive75" / "imagenet_category_classes.csv", "wnid")
    p_lv, a_lv = audit("lvis", A["lvis"]["anchors"], A["lvis"]["exclude"],
                       EXEMPLAR_SETS / "lvis_exclusive104" / "lvis_category_classes.csv", "lvis_category_id")
    pairs = pd.concat([p_in, p_lv], ignore_index=True)
    anchors = pd.concat([a_in, a_lv], ignore_index=True)
    for df in (pairs, anchors):
        df["cdi_semantic"] = df["cdi_category"].map(dict(zip(cdi["category"], cdi["cdi_semantic"])))
        df["in_valid129"] = df["cdi_category"].isin(valid129)
        df["in_valid85"] = df["cdi_category"].isin(valid85)
    pairs = pairs.sort_values(["cdi_category", "dataset", "status", "n_exemplars"], ascending=[True, True, True, False])
    pairs.to_csv(MAPPING_OUT / "mapping_audit_pairs.csv", index=False)
    anchors.to_csv(MAPPING_OUT / "mapping_audit_anchors.csv", index=False)

    words = []
    for w in cdi["category"]:
        row = {"cdi_category": w, "cdi_semantic": dict(zip(cdi["category"], cdi["cdi_semantic"]))[w],
               "in_valid129": w in valid129, "in_valid85": w in valid85,
               "wordnet_first_sense": (wn.synsets(w, "n")[0].name() if wn.synsets(w, "n") else "")}
        for ds, notes in (("imagenet", A["imagenet"]["notes"]), ("lvis", {**A["imagenet"]["notes"], **A["lvis"]["notes"]})):
            pw = pairs[(pairs["dataset"] == ds) & (pairs["cdi_category"] == w)]
            aw = anchors[(anchors["dataset"] == ds) & (anchors["cdi_category"] == w)]
            an = pw[pw["status"] == "analyzed"]
            row[f"{ds}_anchors"] = "; ".join(f"{r.anchor_synset} ({r.anchor_origin})" for r in aw.itertuples())
            row[f"{ds}_analyzed"] = len(an) > 0
            row[f"{ds}_n_classes_analyzed"] = len(an)
            row[f"{ds}_n_exemplars_analyzed"] = int(an["n_exemplars"].sum()) if len(an) else 0
            row[f"{ds}_classes_analyzed"] = "; ".join(f"{r.source_name} ({r.n_exemplars})" for r in an.itertuples())
            row[f"{ds}_excluded"] = "; ".join(f"{r.source_name}: {r.note}"
                                              for r in pw[pw["status"] == "excluded"].itertuples())
            row[f"{ds}_note"] = notes.get(w, "")
        words.append(row)
    pd.DataFrame(words).to_csv(MAPPING_OUT / "mapping_audit_words.csv", index=False)

    for ds in ("imagenet", "lvis"):
        d = pairs[pairs["dataset"] == ds]
        a = anchors[(anchors["dataset"] == ds) & anchors["in_valid129"]]
        print(f"{ds}: pairs analyzed={(d.status == 'analyzed').sum()}, hyponym_only={(d.status == 'hyponym_only').sum()}, "
              f"excluded={(d.status == 'excluded').sum()}; valid129 anchors by origin: "
              f"{a['anchor_origin'].value_counts().to_dict()}")


def build_audit_page():
    pairs = pd.read_csv(MAPPING_OUT / "mapping_audit_pairs.csv")
    anchors = pd.read_csv(MAPPING_OUT / "mapping_audit_anchors.csv")
    words = pd.read_csv(MAPPING_OUT / "mapping_audit_words.csv").fillna("")

    data = []
    for w in words.itertuples():
        entry = {"w": w.cdi_category, "dom": w.cdi_semantic, "v129": bool(w.in_valid129),
                 "v85": bool(w.in_valid85), "first": w.wordnet_first_sense, "ds": {}}
        for ds in ("imagenet", "lvis"):
            a = anchors[(anchors.dataset == ds) & (anchors.cdi_category == w.cdi_category)]
            p = pairs[(pairs.dataset == ds) & (pairs.cdi_category == w.cdi_category)]
            entry["ds"][ds] = {
                "anchors": [[r.anchor_synset, r.anchor_origin, r.anchor_definition] for r in a.itertuples()],
                "rows": [[r.source_name, r.source_synset, None if pd.isna(r.n_exemplars) else int(r.n_exemplars),
                          r.status, "" if pd.isna(r.note) else r.note, int(r.hypernym_distance),
                          r.source_definition]
                         for r in p.sort_values(["status", "n_exemplars"], ascending=[True, False]).itertuples()],
                "note": getattr(w, f"{ds}_note"),
            }
        data.append(entry)

    an = pairs[pairs.status == "analyzed"]
    stats = {}
    for ds in ("imagenet", "lvis"):
        d = an[an.dataset == ds]
        a = anchors[(anchors.dataset == ds) & anchors.in_valid129]
        stats[ds] = {"words": int(d.cdi_category.nunique()), "classes": int(d.source_id.nunique()),
                     "exemplars": int(d.n_exemplars.sum()),
                     "excluded": int((pairs[(pairs.dataset == ds)].status == "excluded").sum()),
                     "origin": a.anchor_origin.value_counts().to_dict()}

    template = (ROOT / "notebooks" / "mapping_audit_template.html").read_text()
    html = template.replace("/*__DATA__*/", json.dumps({"words": data, "stats": stats}, separators=(",", ":")))
    (MAPPING_OUT / "mapping_audit.html").write_text(html)
    print(f"wrote {MAPPING_OUT / 'mapping_audit.html'} ({len(html) / 1e6:.2f} MB)")

## Run (~1 min; reads the LVIS annotations for box counts)

In [5]:
build_audit_tables()
build_audit_page()
pairs = pd.read_csv(MAPPING_OUT / "mapping_audit_pairs.csv")
anchors = pd.read_csv(MAPPING_OUT / "mapping_audit_anchors.csv")

imagenet: pairs analyzed=326, hyponym_only=604, excluded=9; valid129 anchors by origin: {'first_sense': 60, 'other_sense': 20, 'added': 5}
lvis: pairs analyzed=285, hyponym_only=466, excluded=12; valid129 anchors by origin: {'first_sense': 85, 'other_sense': 32, 'added': 7}


wrote /home/j7yang/babyview-projects/vss2026/object-detection/analysis/comparison-datasets/outputs/mapping/mapping_audit.html (0.35 MB)


## How much of the analyzed data depends on hand choices

In [6]:
an = pairs[pairs.status == "analyzed"]
rows = []
for ds, g in an.groupby("dataset"):
    hand = g[g.anchor_origin != "first_sense"]
    first_only = g.groupby("cdi_category").anchor_origin.apply(lambda s: (s == "first_sense").all())
    rows.append({"dataset": ds, "analyzed CDI words": g.cdi_category.nunique(),
                 "words using first-sense anchors only": int(first_only.sum()),
                 "exemplars via hand-chosen anchors": f"{hand.n_exemplars.sum():,.0f} / {g.n_exemplars.sum():,.0f}",
                 "classes excluded by hand": int((pairs[pairs.dataset == ds].status == "excluded").sum())})
pd.DataFrame(rows)

,dataset,analyzed CDI words,words using first-sense anchors only,exemplars via hand-chosen anchors,classes excluded by hand
0,imagenet,75,55,"39,150 / 431,736",9
1,lvis,104,76,"143,517 / 603,785",12


## Senses chosen by hand (valid129)

In [7]:
a = anchors[anchors.in_valid129 & (anchors.anchor_origin != "first_sense")]
a[["dataset", "cdi_category", "anchor_synset", "anchor_origin", "anchor_definition"]].sort_values(["cdi_category", "dataset"])

,dataset,cdi_category,anchor_synset,anchor_origin,anchor_definition
2,imagenet,alligator,alligator.n.02,other_sense,either of two amphibious reptiles related to crocodiles but with shorter broader snouts
157,lvis,alligator,alligator.n.02,other_sense,either of two amphibious reptiles related to crocodiles but with shorter broader snouts
8,imagenet,banana,banana.n.02,other_sense,elongated crescent-shaped yellow fruit with soft sweet flesh
163,lvis,banana,banana.n.02,other_sense,elongated crescent-shaped yellow fruit with soft sweet flesh
323,lvis,belt,belt.n.02,other_sense,a band to tie or buckle around the body (usually at the waist)
...,...,...,...,...,...
300,lvis,table,table.n.02,other_sense,a piece of furniture having a smooth flat top that is usually supported by one or more vertical legs
301,lvis,table,table.n.03,other_sense,a piece of furniture with tableware for a meal laid out on it
308,lvis,toy,toy.n.03,other_sense,a device regarded as providing amusement
151,imagenet,vacuum,vacuum.n.04,other_sense,an electrical home appliance that cleans by suction


## Classes excluded by hand

In [8]:
pairs[pairs.status == "excluded"][["dataset", "cdi_category", "source_name", "n_exemplars", "note"]]

,dataset,cdi_category,source_name,n_exemplars,note
520,imagenet,ball,punching_bag,1297,"punching bag, not a ball"
553,lvis,bed,hammock,23,hammock
705,lvis,box,window_box_(for_plants),278,planter
706,lvis,box,cash_register,40,cash register
709,imagenet,bread,meat_loaf,1350,WordNet files meat loaf under loaf/bread
831,lvis,cat,cougar,7,wild cat
900,imagenet,dish,Petri_dish,1012,lab glassware
1040,imagenet,drink,cup,1350,ImageNet 'cup' images show the container (mapped to cup)
1104,imagenet,food,menu,1350,"a menu, not food"
1105,imagenet,food,hay,1350,animal fodder


## Classes moved to a closer CDI word in the exclusive sets

In [9]:
pairs[(pairs.status == "hyponym_only") & pairs.note.str.startswith("assigned", na=False)][["dataset", "cdi_category", "source_name", "n_exemplars", "note"]]

,dataset,cdi_category,source_name,n_exemplars,note
710,imagenet,bread,pretzel,1350,assigned to closer CDI word: cracker
720,lvis,bread,cracker,622,assigned to closer CDI word: cracker
721,lvis,bread,muffin,506,assigned to closer CDI word: muffin
722,lvis,bread,pretzel,192,assigned to closer CDI word: cracker
906,lvis,dish,bowl,6248,assigned to closer CDI word: bowl
907,lvis,dish,soup_bowl,2,assigned to closer CDI word: bowl
1336,lvis,glove,mitten,189,assigned to closer CDI word: mitten
1440,lvis,napkin,bib,108,assigned to closer CDI word: bib
1461,lvis,pants,jean,6391,assigned to closer CDI word: jeans
1462,lvis,pants,short_pants,6376,assigned to closer CDI word: shorts


The same information as an interactive page: `outputs/mapping/mapping_audit.html` (open in a browser).

### Review checklist

- Is every `other_sense` / `added` anchor the right referent for a child?
- Are the large exclusions (LVIS kites 12,977, vases 6,026) defensible?
- Should COCO sports classes (baseball gloves, jerseys, ski parkas, goggles) be excluded too?